# TP2 - Synchronisation par files de messages
## Files d'attente FreeRTOS

**ECUE : Atelier Systèmes Temps Réel**  
**Support matériel : Arduino AVR + FreeRTOS**

Ce notebook reprend le TP2 sur la création, l'envoi, la réception et le blocage sur une file de messages. Les exemples Arduino sont dans `FreeRTOS_AVR/examples/FreeRTOSBook/`. Les simulations Python permettent d'observer les transitions d'une file sans carte physique.

## Objectifs

- introduire la notion de file de messages ;
- créer une file avec `xQueueCreate()` ;
- envoyer et recevoir des données ;
- comprendre le blocage sur une file pleine ou vide ;
- observer l'effet des priorités lors des opérations de lecture et d'écriture.

Une queue est un tampon partagé entre tâches. Elle possède une longueur maximale, une taille d'élément et des listes d'attente pour les émetteurs et les récepteurs.

## API essentielle

```cpp
QueueHandle_t queue = xQueueCreate(5, sizeof(long));
xQueueSendToBack(queue, &value, xTicksToWait);
xQueueReceive(queue, &value, xTicksToWait);
xQueuePeek(queue, &value, xTicksToWait);
xQueueOverwrite(queue, &value);
uxQueueMessagesWaiting(queue);
```

`xTicksToWait` définit le temps maximal d'attente lorsqu'une queue est pleine pour un émetteur ou vide pour un récepteur. `portMAX_DELAY` correspond à une attente indéfinie.

## API complémentaires des files

Cette version AVR de FreeRTOS fournit également :

```cpp
xQueueSendToFront(queue, &value, xTicksToWait);
xQueuePeek(queue, &value, xTicksToWait);
xQueueOverwrite(queue, &value);
uxQueueSpacesAvailable(queue);
xQueueReset(queue);
```

- `xQueueSendToFront()` insère un élément au début de la file ;
- `xQueuePeek()` lit le premier élément sans le supprimer ;
- `xQueueOverwrite()` remplace la valeur conservée dans une queue de longueur 1 ;
- `uxQueueSpacesAvailable()` retourne le nombre de cases libres ;
- `xQueueReset()` vide la file et réinitialise son état.

Depuis une interruption, utiliser uniquement les variantes `FromISR`, par exemple `xQueueSendToBackFromISR()`, `xQueueReceiveFromISR()` et `xQueueOverwriteFromISR()`. Le paramètre `pxHigherPriorityTaskWoken` signale qu'une tâche plus prioritaire a été débloquée.

**Attention :** `xQueueOverwrite()` n'est pas un remplacement général de `xQueueSendToBack()`. Dans cette version du noyau, elle est destinée aux queues de longueur 1.

In [1]:
from collections import deque


class FileMessages:
    """Modele pedagogique d'une queue FreeRTOS de longueur fixe."""

    def __init__(self, capacite):
        if capacite < 1:
            raise ValueError('La capacite doit etre positive.')
        self.capacite = capacite
        self.messages = deque()

    def envoyer(self, message):
        """Equivalent simplifie de xQueueSendToBack(..., 0)."""
        if self.est_pleine():
            return False
        self.messages.append(message)
        return True

    def envoyer_en_tete(self, message):
        """Equivalent simplifie de xQueueSendToFront(..., 0)."""
        if self.est_pleine():
            return False
        self.messages.appendleft(message)
        return True

    def ecraser(self, message):
        """Equivalent pedagogique de xQueueOverwrite pour une queue de longueur 1."""
        if self.capacite != 1:
            raise ValueError('xQueueOverwrite est reserve aux queues de longueur 1.')
        if self.messages:
            self.messages[0] = message
        else:
            self.messages.append(message)
        return True

    def recevoir(self):
        """Equivalent simplifie de xQueueReceive(..., 0)."""
        if self.est_vide():
            return None
        return self.messages.popleft()

    def consulter(self):
        """Equivalent simplifie de xQueuePeek(..., 0), sans suppression."""
        return None if self.est_vide() else self.messages[0]

    def reinitialiser(self):
        self.messages.clear()

    def est_vide(self):
        return len(self.messages) == 0

    def est_pleine(self):
        return len(self.messages) >= self.capacite

    def espaces_disponibles(self):
        return self.capacite - len(self.messages)

    def etat(self):
        return {
            'messages': list(self.messages),
            'utilises': len(self.messages),
            'capacite': self.capacite,
            'espaces_disponibles': self.espaces_disponibles(),
        }


queue = FileMessages(3)
operations = [
    queue.envoyer(100),
    queue.envoyer(200),
    queue.envoyer_en_tete(50),
    queue.consulter(),
    queue.envoyer(300),
    queue.recevoir(),
]
operations, queue.etat()

([True, True, True, 50, False, 50],
 {'messages': [100, 200],
  'utilises': 2,
  'capacite': 3,
  'espaces_disponibles': 1})

## Exercice 1 - Création, envoi et réception

Ouvrir `Example010`, téléverser le programme et observer `Sender1`, `Sender2` et `Receiver`. La queue contient au maximum 5 valeurs de type `long`.

**Questions :**

1. Quelle est la taille de la queue ?
2. Expliquer le comportement de l'application.
3. À quoi sert `xTicksToWait` dans `vReceiverTask` ?
4. Provoquer les messages `Could not send to the queue`, `Queue should have been empty!` et `Could not receive from the queue` en modifiant les priorités, les délais, la capacité ou le temps d'attente.
5. Ajouter `Sender3`, `Sender4` et `Sender5` en réutilisant la même fonction.

Référence : [`Example010.ino`](FreeRTOS_AVR/examples/FreeRTOSBook/Example010/Example010.ino).

## Exercice 2 - File de messages de type composé

Ouvrir `Example011`. Une structure `xData` regroupe la valeur envoyée et l'identifiant de son émetteur.

```cpp
typedef struct {
    unsigned char ucValue;
    unsigned char ucSource;
} xData;

QueueHandle_t queue = xQueueCreate(3, sizeof(xData));
```

Observer le comportement des deux émetteurs de priorité 2 et du récepteur de priorité 1. Expliquer pourquoi la queue devient pleine avant que le récepteur ne s'exécute.

Référence : [`Example011.ino`](FreeRTOS_AVR/examples/FreeRTOSBook/Example011/Example011.ino).

In [6]:
def simulation_producteurs(capacite, producteurs, lectures):
    queue = FileMessages(capacite)
    journal = []
    for valeur in producteurs:
        journal.append((f'Sender{valeur}', queue.envoyer(valeur), queue.etat()['utilises']))
    for _ in range(lectures):
        journal.append(('Receiver', queue.recevoir(), queue.etat()['utilises']))
    return journal

simulation_producteurs(3, [100, 200, 100, 200, 100], 4)

[('Sender100', True, 1),
 ('Sender200', True, 2),
 ('Sender100', True, 3),
 ('Sender200', False, 3),
 ('Sender100', False, 3),
 ('Receiver', 100, 2),
 ('Receiver', 200, 1),
 ('Receiver', 100, 0),
 ('Receiver', None, 0)]

In [2]:
def experimenter_blocage(capacite, emissions, lectures):
    queue = FileMessages(capacite)
    journal = []

    for message in emissions:
        ajoute = queue.envoyer(message)
        journal.append({
            'operation': 'send',
            'message': message,
            'succes': ajoute,
            'etat': queue.etat(),
        })

    for _ in range(lectures):
        avant = queue.etat()
        message = queue.consulter()
        recu = queue.recevoir()
        journal.append({
            'operation': 'peek_then_receive',
            'peek': message,
            'receive': recu,
            'etat_avant': avant,
            'etat_apres': queue.etat(),
        })

    return journal

experimenter_blocage(2, ['A', 'B', 'C'], 3)

[{'operation': 'send',
  'message': 'A',
  'succes': True,
  'etat': {'messages': ['A'],
   'utilises': 1,
   'capacite': 2,
   'espaces_disponibles': 1}},
 {'operation': 'send',
  'message': 'B',
  'succes': True,
  'etat': {'messages': ['A', 'B'],
   'utilises': 2,
   'capacite': 2,
   'espaces_disponibles': 0}},
 {'operation': 'send',
  'message': 'C',
  'succes': False,
  'etat': {'messages': ['A', 'B'],
   'utilises': 2,
   'capacite': 2,
   'espaces_disponibles': 0}},
 {'operation': 'peek_then_receive',
  'peek': 'A',
  'receive': 'A',
  'etat_avant': {'messages': ['A', 'B'],
   'utilises': 2,
   'capacite': 2,
   'espaces_disponibles': 0},
  'etat_apres': {'messages': ['B'],
   'utilises': 1,
   'capacite': 2,
   'espaces_disponibles': 1}},
 {'operation': 'peek_then_receive',
  'peek': 'B',
  'receive': 'B',
  'etat_avant': {'messages': ['B'],
   'utilises': 1,
   'capacite': 2,
   'espaces_disponibles': 1},
  'etat_apres': {'messages': [],
   'utilises': 0,
   'capacite': 2,
  

## Exercice 3 - Trame avec acquittement

Deux tâches communiquent avec une trame de 1 à 16 octets. L'émetteur envoie une longueur et les données, puis attend une réponse contenant le nombre d'octets et le code `ACK` ou `NACK`.

```text
Emetteur  -- n, Data[n] -->  Récepteur
Emetteur  <-- n, ACK -----  Récepteur
```

**Travail demandé :** implémenter les deux tâches FreeRTOS. L'émetteur ne doit envoyer la trame suivante qu'après réception de l'acquittement correspondant.

In [5]:
def protocole_ack(messages):
    resultats = []
    for numero, donnees in enumerate(messages, start=1):
        if not 1 <= len(donnees) <= 16:
            resultats.append((numero, 'NACK', len(donnees)))
            continue
        resultats.append((numero, 'ACK', len(donnees)))
    return resultats

protocole_ack([b'RTOS', b'FreeRTOS', b''])

[(1, 'ACK', 4), (2, 'ACK', 8), (3, 'NACK', 0)]

## `taskYIELD()`, blocage et compte rendu

`taskYIELD()` demande au noyau de rechercher une autre tâche prête. Si aucune tâche de priorité supérieure ou égale n'est prête, la tâche appelante peut être resélectionnée.

### Expérience sur `xTicksToWait`

Réaliser au moins trois essais dans `Example010` :

| Situation | Paramètre à modifier | Observation attendue |
|---|---|---|
| Queue pleine à l'envoi | capacité, priorité ou vitesse des Sender | `xQueueSendToBack()` retourne `errQUEUE_FULL` si l'attente vaut 0 |
| Queue vide à la réception | priorité du Receiver ou période des Sender | la tâche reçoit `pdFALSE` après expiration du délai |
| Attente bloquante | `xTicksToWait = portMAX_DELAY` | la tâche reste bloquée jusqu'à l'arrivée d'un message ou d'un espace |

Ne pas confondre **timeout** et **erreur système** : un retour `pdFALSE` peut simplement signifier que le délai est expiré.

Pour le compte rendu, joindre les observations du moniteur série, les paramètres modifiés, les états de la queue et une explication du rôle des priorités et des temps d'attente. Ajouter la capacité, la taille d'un élément et le nombre de messages en attente à chaque scénario.